# Local pipeline (all-in-one)

Same as running each `NB_*.ipynb`. Prefer `%run ./common/bootstrap` per notebook in Fabric.


In [ ]:
# Prefer repo bootstrap (read_table, cache, reload_io_helpers stay in sync).
%run ./common/bootstrap
init_notebook(globals())


In [ ]:
import os

# Set before OneLake reads (re-run this cell to change without restarting the kernel).
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
os.environ.setdefault("MEDALLION_SCAN_RETRIES", "6")
# os.environ["MEDALLION_DEBUG_READ"] = "1"
# os.environ["MEDALLION_CACHE_REFRESH"] = "1"  # force OneLake; ignore local parquet cache

print(
    "Local read settings:",
    f"cache={os.environ.get('MEDALLION_LOCAL_TABLE_CACHE')}",
    f"chunk_rows={os.environ.get('MEDALLION_SCAN_CHUNK_ROWS')}",
    f"retries={os.environ.get('MEDALLION_SCAN_RETRIES')}",
    f"debug={os.environ.get('MEDALLION_DEBUG_READ', '0')}",
    f"refresh={os.environ.get('MEDALLION_CACHE_REFRESH', '0')}",
)


## NB_HCHistorical_Bronze_To_Silver.ipynb


In [ ]:
bronze = read_table(TABLE_HC_BRONZE)
save_table_cache(bronze, TABLE_HC_BRONZE)
country_lookup = load_country_lookup(spark, TABLE_COUNTRY)


In [ ]:
hc = bronze.select(
    F.col("`Month`").cast("int").alias("Month_No"),
    F.col("`Person ID`").alias("OktaUserEmployeeNumber"),
    F.col("`Employee Code`").alias("HCEmployeeNumber"),
    F.col("`Fullname [Employee]`").alias("OktaUserFullName"),
    F.col("`Region [Employee]`").alias("HCRegion"),
    F.col("`Country [Employee]`").alias("_raw_country"),
    F.col("`Office [Employee]`").alias("HCOffice"),
    F.col("`Business Unit [Employee]`").alias("HCBusinessUnit"),
    F.col("`Division [Employee]`").alias("HCDivision"),
    F.col("`Department [Employee]`").alias("HCDepartment"),
    F.col("`Legal Entity [Employee]`").alias("HCLegalEntity"),
    F.col("`Grid Subgroup [Employee]`").alias("HCGridSubgroup"),
    F.col("`Talent Segment [Employee]`").alias("HCTalentSegment"),
    F.col("`Job Title [Employee]`").alias("HCJobTitle"),
    F.lower(F.trim(F.col("`Email [Employee]`"))).alias("OktaUserEmail"),
    F.col("`User Network [Employee]`").alias("HCUserNetwork"),
    F.col("`Inc. Roles`").alias("HCIncRoles"),
    F.col("`Okta Region`").alias("OktaUserRegion"),
    F.col("`Division INC`").alias("HCDivisionINC"),
    F.col("`Total Employees`").cast("int").alias("HCTotalEmployees"),
    F.col("`Area Emeal`").alias("HCAreaEmeal"),
    F.col("KeyIdEmployeeCountry").alias("HCKeyIdEmployeeCountry"),
    F.col("`Business Unit Code [Employee]`").alias("HCBusinessUnitCode"),
    F.col("`Replace Mapping`").alias("HCReplaceMapping"),
)

hc = with_normalized_country(hc, "_raw_country", country_lookup).drop("_raw_country")

hc = hc.withColumn("HCDate", date_from_yyyymm(F.col("Month_No")))
hc = hc.withColumn(
    "HCKeyDivision",
    F.concat(F.col("HCDivisionINC"), F.lit("/"), F.col("OktaUserCountry")),
)
hc = hc.withColumn(
    "HCKeyEmail",
    F.concat(F.col("Month_No").cast("string"), F.lit("/"), F.col("OktaUserEmail")),
)

# Business keys: Month_No + OktaUserEmail should be unique per spec; enforce not null on required fields
hc = hc.filter(
    F.col("Month_No").isNotNull()
    & F.col("OktaUserEmail").isNotNull()
    & F.col("HCDate").isNotNull()
    & F.col("HCKeyDivision").isNotNull()
    & F.col("HCKeyEmail").isNotNull()
)

In [ ]:
write_full_table(hc, TABLE_HC_SILVER)

show_sample(hc.limit(10))

## NB_OktaUserforAI_Bronze_To_Silver.ipynb


In [ ]:
bronze = read_table(TABLE_OKTA_BRONZE)
save_table_cache(bronze, TABLE_OKTA_BRONZE)
country_lookup = load_country_lookup(spark, TABLE_COUNTRY)

login_norm = F.lower(F.trim(F.col("login")))
email_norm = F.lower(F.trim(F.col("email")))


In [ ]:
okta = bronze.select(
    F.col("id").alias("OktaUserId"),
    F.trim(F.col("display_name")).alias("OktaUserFullName"),
    F.trim(F.col("first_name")).alias("OktaUserFirstName"),
    F.trim(F.col("last_name")).alias("OktaUserLastName"),
    login_norm.alias("OktaUserEmail"),
    email_norm.alias("OktaUserFullEmail"),
    F.trim(F.col("employee_number")).alias("OktaUserEmployeeNumber"),
    F.trim(F.col("region")).alias("OktaUserRegion"),
    F.col("okta_id").alias("OktaUserOktaId"),
    F.col("updated_at").cast("timestamp").alias("OktaUserUpdatedAt"),
    F.col("deleted_at").cast("timestamp").alias("OktaUserDeletedAt"),
    F.trim(F.col("country")).alias("_raw_country"),
    F.trim(F.col("okta_tenant")).alias("OktaUserTenant"),
)

okta = okta.filter(
    F.col("OktaUserId").isNotNull()
    & is_valid_email(F.col("OktaUserEmail"))
    & is_valid_email(F.col("OktaUserFullEmail"))
    & (F.lower(F.col("OktaUserRegion")) != F.lit("japan-onentt"))
    & (F.lower(F.col("OktaUserTenant")) != F.lit("connect-onentt"))
)

okta = with_normalized_country(okta, "_raw_country", country_lookup).drop("_raw_country")

In [ ]:
write_full_table(okta, TABLE_OKTA_SILVER)

show_sample(okta.limit(10))

## NB_CursorUsage_Bronze_To_Silver.ipynb


In [ ]:
bronze_df = read_table(TABLE_CURSOR_BRONZE)
save_table_cache(bronze_df, TABLE_CURSOR_BRONZE)
if CURSOR_BRONZE_FILENAME:
    bronze_df = bronze_df.filter(F.col("FileName") == CURSOR_BRONZE_FILENAME)
bronze = bronze_df

okta = read_table(TABLE_OKTA_SILVER)
save_table_cache(okta, TABLE_OKTA_SILVER)
hc = read_table(TABLE_HC_SILVER)
save_table_cache(hc, TABLE_HC_SILVER)

user_norm = F.lower(F.trim(F.col("User")))



In [ ]:
base = (
    bronze.filter(is_valid_email(F.col("User")))
    .select(
        F.col("Date").cast("date").alias("CursorUsageDate"),
        user_norm.alias("CursorUsageUser"),
        F.col("`Service Account Name`").alias("CursorUsageServiceAccountName"),
        F.col("`Service Account ID`").alias("CursorUsageServiceAccountID"),
        F.col("`Cloud Agent ID`").alias("CursorUsageCloudAgentID"),
        F.col("`Automation ID`").alias("CursorUsageAutomationID"),
        F.col("Kind").alias("CursorUsageKind"),
        F.col("Model").alias("CursorUsageModel"),
        F.col("`Max Mode`").alias("CursorUsageMaxMode"),
        F.col("`Input (w/ Cache Write)`").cast("long").alias("CursorUsageInputWithCacheWrite"),
        F.col("`Input (w/o Cache Write)`").cast("long").alias("CursorUsageInputWithoutCacheWrite"),
        F.col("`Cache Read`").cast("long").alias("CursorUsageCacheRead"),
        F.col("`Output Tokens`").cast("long").alias("CursorUsageOutputTokens"),
        F.col("`Total Tokens`").cast("long").alias("CursorUsageTotalTokens"),
        F.col("Cost").cast("decimal(18,8)").alias("CursorUsageCost"),
        F.col("FileName").alias("CursorUsageFileName"),
        F.col("ExecutionDate").cast("timestamp").alias("CursorUsageIngestionDate"),
    )
    .withColumn(
        "CursorUsageMonthNo",
        (F.year("CursorUsageDate") * F.lit(100) + F.month("CursorUsageDate")).cast("int"),
    )
)

okta_slim = okta.select(
    "OktaUserId",
    "OktaUserEmail",
    "OktaUserFullEmail",
    "OktaUserEmployeeNumber",
    "OktaUserFullName",
    "OktaUserRegion",
    "OktaUserCountry",
)

# Inner join: match login or full email
joined = base.join(
    okta_slim,
    (F.col("CursorUsageUser") == F.col("OktaUserEmail"))
    | (F.col("CursorUsageUser") == F.col("OktaUserFullEmail")),
    "inner",
)

joined = joined.withColumn(
    "CursorUsageKeyEmail",
    F.when(
        F.col("OktaUserEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserEmail"),
        ),
    ),
).withColumn(
    "CursorUsageKeyFullEmail",
    F.when(
        F.col("OktaUserFullEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserFullEmail"),
        ),
    ),
)

hc_slim = hc.select(
    F.col("HCKeyEmail"),
    F.col("HCBusinessUnit"),
    F.col("HCBusinessUnitCode"),
    F.col("HCDivision"),
    F.col("HCJobTitle"),
    F.col("HCOffice"),
    F.col("HCTalentSegment"),
    F.col("Month_No").alias("HCMonthNo"),
    F.col("OktaUserCountry").alias("HCOktaUserCountry"),
    F.col("HCDivisionINC"),
)

joined = joined.join(
    hc_slim,
    (F.col("HCKeyEmail") == F.col("CursorUsageKeyEmail"))
    | (F.col("HCKeyEmail") == F.col("CursorUsageKeyFullEmail")),
    "left",
).withColumn(
    "HCDivisionINC",
    F.coalesce(F.col("HCDivisionINC"), F.lit(HC_DIVISION_INC_DEFAULT)),
)

write_df= joined.distinct()

write_df=write_df.withColumn(
    FACT_CURSOR_ACTIVE_MERGE_KEY,
    cursor_usage_record_key()
    )

In [ ]:
merge_incremental(
    spark,
    write_df,
    TABLE_CURSOR_SILVER,
    FACT_CURSOR_ACTIVE_MERGE_KEY,
)

## NB_CursorUsage_Gold.ipynb


In [ ]:
silver = read_table(TABLE_CURSOR_SILVER)
save_table_cache(silver, TABLE_CURSOR_SILVER)

request = F.lit(1).cast("int")
use_case_multiplier = F.lit(CURSOR_USE_CASE_MULTIPLIER).cast("decimal(18,8)")
automation_poc = F.lit(CURSOR_AUTOMATION_POC).cast("decimal(18,8)")
total_hours_saved_poc = F.lit(CURSOR_TOTAL_HOURS_SAVED_POC).cast("decimal(18,8)")


In [ ]:
fact = silver.select(
    F.col("CursorUsageMonthNo").alias("Month_No"),
    request.alias("Request"),
    F.lit(CURSOR_ASSET_ID).alias("assetId"),
    F.lit(CURSOR_ASSET_NAME).alias("assetName"),
    F.lit(None).cast("string").alias("eventType"),
    F.lit(None).cast("string").alias("id_project"),
    F.lit(None).cast("string").alias("project"),
    F.lit(None).cast("string").alias("account"),
    F.col("OktaUserId").alias("userId"),
    F.col("OktaUserFullName").alias("username"),
    F.col("OktaUserEmployeeNumber").alias("employeeid"),
    F.col("OktaUserEmail").alias("userEmail"),
    F.col("OktaUserFullEmail").alias("userFullEmail"),
    F.col("OktaUserRegion").alias("Division"),
    F.col("OktaUserCountry").alias("userCountry"),
    F.lit(CURSOR_USE_CASE_CODE).alias("useCaseCode"),
    F.lit(CURSOR_USE_CASE_DISPLAY_NAME).alias("useCaseDisplayName"),
    (request.cast("decimal(18,8)") * total_hours_saved_poc * use_case_multiplier).alias(
        "UseCaseHoursSaved"
    ),
    (
        request.cast("decimal(18,8)")
        * (total_hours_saved_poc / automation_poc)
    ).alias("UseCaseTotalHours"),
    F.lit(1).cast("int").alias("UserWithActivity"),
    F.col("CursorUsageDate").alias("LastModification"),
    F.col("HCBusinessUnit").alias("User_UN"),
    F.col("HCJobTitle").alias("JobRoles"),
    F.col("HCOktaUserCountry").alias("Employee_Country"),
    F.col("HCOffice").alias("Employee_Office"),
    F.col("HCTalentSegment").alias("Segment"),
    F.col("HCDivision").alias("Foundation"),
    F.col("HCBusinessUnitCode").alias("Employee_Business_Unit_Code"),
    F.lit(None).cast("string").alias("asset_model"),
    F.lit(None).cast("string").alias("asset_useCase"),
    F.lit(None).cast("string").alias("asset_mySourceType"),
    F.lit(None).cast("string").alias("asset_sourceType"),
    F.lit(None).cast("string").alias("billingClientName"),
    F.lit(None).cast("string").alias("billingCountry"),
    F.lit(None).cast("string").alias("billingRegion"),
    F.lit(None).cast("string").alias("flowid"),
    use_case_multiplier.alias("useCaseMultiplier"),
    automation_poc.alias("automation_poc"),
    total_hours_saved_poc.alias("totalHoursSaved_poc"),
    F.col(FACT_CURSOR_ACTIVE_MERGE_KEY),
)

# Required not-null columns per spec
fact = fact.filter(
    F.col("Month_No").isNotNull()
    & F.col("Request").isNotNull()
    & F.col("assetId").isNotNull()
    & F.col("assetName").isNotNull()
    & F.col("useCaseCode").isNotNull()
    & F.col("useCaseDisplayName").isNotNull()
    & F.col("UseCaseHoursSaved").isNotNull()
    & F.col("UseCaseTotalHours").isNotNull()
    & F.col("UserWithActivity").isNotNull()
    & F.col("LastModification").isNotNull()
    & F.col("useCaseMultiplier").isNotNull()
    & F.col("automation_poc").isNotNull()
    & F.col("totalHoursSaved_poc").isNotNull()
    & F.col(FACT_CURSOR_ACTIVE_MERGE_KEY).isNotNull()
)

In [ ]:
merge_incremental(
    spark,
    fact,
    TABLE_FACT_CURSOR_ACTIVE,
    FACT_CURSOR_ACTIVE_MERGE_KEY,
)

show_sample(fact.limit(10))